# Extend Your Chatbot

In the Level 1 notebook you built a working chatbot. Now let's extend it with new API features — each section adds one capability:

1. **Verbosity** — control how concise or detailed the response is
2. **Max Output Tokens** — cap the response length
3. **Truncation** — handle long conversations gracefully
4. **Tool Calling** — let the AI call functions to get real data

Each section is self-contained: read the explanation, run the cell, see the result.

> **Have a coding agent?** These exercises also exist as [PRDs](PRDs/) you can paste into Claude Code or another coding agent to have it implement the changes for you.

In [ ]:
# Install required packages into the Jupyter kernel
# (only needed once — safe to re-run)
%pip install openai python-dotenv httpx

## Setup

Same setup as the Level 1 notebook — load credentials and create the client.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

env_path = Path.cwd() / ".env"
load_dotenv(env_path)

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL = os.getenv("MODEL_NAME", "gpt-5.2")
REASONING_EFFORT = os.getenv("REASONING_EFFORT", "low")
INSTRUCTIONS = os.getenv(
    "CHATBOT_INSTRUCTIONS",
    "You are a helpful assistant. Be concise and friendly.",
)

if not ENDPOINT or not API_KEY:
    print("ERROR: Missing configuration!")
    print(f"  Looked for .env at: {env_path}")
    print("  Make sure AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY are set.")
else:
    client = OpenAI(base_url=ENDPOINT, api_key=API_KEY, max_retries=10)
    print("Ready!")

---

## 1. Verbosity

The Responses API has a `verbosity` setting that controls how concise or detailed the response is. This is separate from `reasoning_effort`:

- **`reasoning_effort`** controls how much internal thinking the model does
- **`verbosity`** controls how long and detailed the visible answer is

**Important:** `verbosity` goes under the `text` parameter, not at the top level.

```python
text={"verbosity": "low"}   # low, medium, or high
```

Let's see the difference between `low` and `high`.

In [ ]:
message = "Explain how machine learning works."

# --- Low verbosity ---
response_low = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    text={"verbosity": "low"},
)

print("=== Verbosity: LOW ===")
print(f"{response_low.output_text}\n")

# --- High verbosity ---
response_high = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    text={"verbosity": "high"},
)

print("=== Verbosity: HIGH ===")
print(f"{response_high.output_text}")

---

## 2. Max Output Tokens

`max_output_tokens` sets a hard cap on how many tokens the model can generate. Unlike verbosity (which is a hint), this is a **strict limit** — the response will be cut off if it hits the cap.

This is useful for controlling cost, keeping demos snappy, or preventing runaway responses.

Unlike `verbosity`, this is a **top-level** parameter:

```python
max_output_tokens=150   # any positive integer
```

Let's see how it affects a response that would normally be long.

In [ ]:
message = "Write a detailed explanation of how the internet works."

# --- With a small token cap ---
response_short = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    max_output_tokens=50,
)

print("=== max_output_tokens=50 ===")
print(f"{response_short.output_text}")
if response_short.usage:
    print(f"  [output tokens used: {response_short.usage.output_tokens}]")
print()

# --- With a larger cap ---
response_long = client.responses.create(
    model=MODEL,
    input=message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    max_output_tokens=300,
)

print("=== max_output_tokens=300 ===")
print(f"{response_long.output_text}")
if response_long.usage:
    print(f"  [output tokens used: {response_long.usage.output_tokens}]")

---

## 3. Truncation

Every AI model has a **context window** — a limit on how much text it can process at once. As a conversation grows, the accumulated messages consume more of this window. When it's exceeded, the API returns an error.

The Responses API has a built-in `truncation` parameter that handles this automatically:

- `truncation="disabled"` (default) — returns an error if the context is exceeded
- `truncation="auto"` — automatically drops the oldest messages to stay within limits

This is a one-line safety net:

```python
truncation="auto"
```

You won't hit the limit in this notebook, but let's verify the parameter works by using it in a multi-turn conversation.

In [ ]:
# Turn 1
response1 = client.responses.create(
    model=MODEL,
    input="What is the capital of France?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    truncation="auto",  # <-- One-line safety net
)

print(f"You:       What is the capital of France?")
print(f"Assistant: {response1.output_text}\n")

# Turn 2 — chained with truncation enabled
response2 = client.responses.create(
    model=MODEL,
    input="What's the population?",
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    previous_response_id=response1.id,
    truncation="auto",
)

print(f"You:       What's the population?")
print(f"Assistant: {response2.output_text}")
if response2.usage:
    print(f"\n  [tokens: {response2.usage.input_tokens} in, {response2.usage.output_tokens} out]")
    print(f"  Context window is large, so no truncation happened here.")
    print(f"  But if this were a very long conversation, the oldest messages would be dropped automatically.")

---

## 4. Tool Calling (Function Calling)

AI models can do more than generate text — they can **call functions** you define. This lets the model reach out to APIs, databases, or any code you write to get real data before responding.

### How it works

```
User: "What's the weather in Durham?"
  → Model sees your tool definitions and decides to call get_weather(location="Durham, NC")
  → Your code runs the function and returns real weather data
  → Model writes a natural language response using that data
```

The key insight: **the model decides _when_ to call a tool and _what arguments_ to pass, but your code provides the implementation.**

There are 3 pieces:
1. **Tool definition** — a JSON schema telling the model what functions are available
2. **Function implementation** — the actual code that runs when the tool is called
3. **Tool-calling loop** — detect function calls in the response, run them, send results back

### Step 1: Define the tool and implement the function

The tool definition tells the model: "there's a function called `get_weather` that takes a `location` string." The model uses this to decide when to call it and what arguments to pass.

The function itself calls [Open-Meteo](https://open-meteo.com/), a free weather API that needs no API key.

In [ ]:
import httpx

# --- Tool definition (JSON schema) ---
# This is what the model sees. It describes the function's name,
# purpose, and parameters so the model knows when and how to call it.
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a given location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g. 'Durham, NC'",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


# --- Function implementation ---
# This is what YOUR CODE runs when the model calls the tool.
# The model never sees this code — it only sees the JSON result.
def get_weather(location):
    """Fetch real weather data from Open-Meteo (free, no API key needed)."""
    # Strip state/country after comma — Open-Meteo geocoder works better with just the city
    city_name = location.split(",")[0].strip()

    # Step 1: Convert city name to coordinates
    geo = httpx.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city_name, "count": 1},
    ).json()

    if not geo.get("results"):
        return json.dumps({"error": f"Could not find location: {location}"})

    place = geo["results"][0]

    # Step 2: Get current weather at those coordinates
    weather = httpx.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
        },
    ).json()

    current = weather["current"]
    return json.dumps({
        "location": place.get("name", location),
        "temperature": f"{current['temperature_2m']}°F",
        "humidity": f"{current['relative_humidity_2m']}%",
        "wind_speed": f"{current['wind_speed_10m']} mph",
    })


# Map tool names to functions so we can look them up dynamically
tool_mapping = {
    "get_weather": get_weather,
}

print("Tool defined: get_weather")
print("Try it directly:")
print(get_weather("Durham, NC"))

### Step 2: The tool-calling loop

When you pass `tools=tools` to the API, the model might respond with a `function_call` instead of text. When that happens, you:

1. Run the function with the arguments the model chose
2. Send the result back as a `function_call_output`
3. Let the model generate a natural language response using that data

The model might call tools **multiple times** (e.g., "compare the weather in Durham and San Francisco"), so we loop until there are no more tool calls.

In [ ]:
# Ask about the weather — the model will call our tool!
user_message = "What's the weather like in Durham, NC?"

response = client.responses.create(
    model=MODEL,
    input=user_message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    tools=tools,  # <-- Tell the model about our tools
)

# --- Tool-calling loop ---
while True:
    tool_outputs = []
    for item in response.output:
        if item.type == "function_call":
            # The model wants to call a function!
            print(f"  [Model is calling: {item.name}({item.arguments})]")
            args = json.loads(item.arguments)
            func = tool_mapping.get(item.name)
            result = func(**args) if func else json.dumps({"error": f"Unknown tool: {item.name}"})
            print(f"  [Result: {result}]")
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": result,
            })

    if not tool_outputs:
        break  # No tool calls — the model gave a text response

    # Send the tool results back so the model can write its answer
    response = client.responses.create(
        model=MODEL,
        input=tool_outputs,
        previous_response_id=response.id,
        tools=tools,
    )

print(f"\nYou:       {user_message}")
print(f"Assistant: {response.output_text}")

### Try it: Multiple tool calls

The model can call the same tool multiple times in one turn. Try asking it to compare the weather in two cities!

In [ ]:
# Try comparing two cities — the model should call get_weather twice!
user_message = "Compare the weather in Durham, NC and San Francisco, CA."

response = client.responses.create(
    model=MODEL,
    input=user_message,
    instructions=INSTRUCTIONS,
    reasoning={"effort": REASONING_EFFORT},
    tools=tools,
)

# Same tool-calling loop as before
while True:
    tool_outputs = []
    for item in response.output:
        if item.type == "function_call":
            print(f"  [Calling: {item.name}({item.arguments})]")
            args = json.loads(item.arguments)
            func = tool_mapping.get(item.name)
            result = func(**args) if func else json.dumps({"error": f"Unknown tool: {item.name}"})
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": result,
            })

    if not tool_outputs:
        break

    response = client.responses.create(
        model=MODEL,
        input=tool_outputs,
        previous_response_id=response.id,
        tools=tools,
    )

print(f"\nYou:       {user_message}")
print(f"Assistant: {response.output_text}")

---

## You Extended Your Chatbot!

Here's what you added:

| Feature | API Parameter | What it does |
|---------|--------------|-------------|
| **Verbosity** | `text={"verbosity": "low"}` | Controls how concise or detailed the response is |
| **Max Output Tokens** | `max_output_tokens=150` | Hard cap on response length |
| **Truncation** | `truncation="auto"` | Automatically drops old messages when context is exceeded |
| **Tool Calling** | `tools=[...]` | Lets the model call functions to get real data |

### What's next?

- **Try adding your own tools** — any function that returns a JSON string can be a tool. Ideas: calculator, dictionary lookup, database query
- **[Level 3 — Web UI](level-3-web/)** — build a browser-based chat with streaming and file uploads
- **Combine features** — add `verbosity`, `max_output_tokens`, `truncation`, and `tools` to the same API call for a fully-featured chatbot